# 把 PyBullet 渲染搬上 GPU：在真实 episode 上验证三步计划

<a href="https://colab.research.google.com/github/yangyi02/droid/blob/main/notebooks/pybullet_gpu_pipeline_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

前一个 notebook（[pybullet_egl_mask_benchmark.ipynb](pybullet_egl_mask_benchmark.ipynb)）
在一个合成 pose 上证明了：EGL 光栅化器快 4.4 倍，但 `alpha=0` 隐藏在它下面失效，
必须改成从 URDF 里删几何。这个 notebook 拿**真实 DROID episode** 验证三件事：

| | 主张 | 怎么算验过 |
|---|---|---|
| **第 1 步** | `PyBulletRenderer(gpu=True)` 在真实数据上与 CPU 等价且更快 | 三台相机、多帧的 mask IoU / 深度分位数 / 每帧耗时 |
| **第 2 步** | 点云生成可以降分辨率，换更大的加速 | 各分辨率下的点数、耗时，以及世界点云是否还是同一朵 |
| **第 3 步** | 基于 PyBullet 的外参优化可以取代 yourdfpy | 同一 episode 上两条路各跑一遍，用 `compute_metrics.evaluate_extrinsics` 这把中立尺子量 |

三步是有依赖顺序的：第 2 步依赖第 1 步的 `gpu=True`，第 3 步依赖前两步。
所以下面按顺序走，前一步不过关就没必要看后一步。

---

### 跑之前

**1. 需要 GPU runtime**，否则 EGL 加载不上，全部退回 CPU，对照不成立。第 0 节会明说。

**2. 需要一个真实 episode 的 Stage 1 输出**（depth + robot + calibration）。
本地 checkout 会直接用 `data/cache/depth/`；Colab 上从
`gs://dm-tapnet/mv-tap/droid/` 拉一个 episode（需要 `gcloud` 认证，约 1-2 GB）。

**3. 会编译一次 PyBullet**（要 NumPy 支持），Colab 上十几分钟。已装好则跳过。

## 0. 环境

In [ ]:
# @title 0a. 找到仓库
import os
import subprocess
import sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_DIR = "/content/droid"
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "--recursive",
                        "https://github.com/yangyi02/droid.git", REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    REPO_DIR = os.getcwd()
    while not os.path.exists(os.path.join(REPO_DIR, "core", "physics.py")):
        parent = os.path.dirname(REPO_DIR)
        if parent == REPO_DIR:
            REPO_DIR = os.getcwd()
            break
        REPO_DIR = parent

if not os.path.exists(os.path.join(REPO_DIR, "core", "physics.py")):
    raise SystemExit(f"{REPO_DIR} is not the droid repo root -- open this "
                     "notebook from the checkout, or set REPO_DIR by hand.")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

CACHE_DIR = os.path.join(REPO_DIR, "data", "cache")
print(f"{'Colab' if IN_COLAB else 'Local'}: {REPO_DIR}")

In [ ]:
# @title 0b. PyBullet + 依赖
# The NumPy build matters here for the same reason as in the previous notebook:
# without it every getCameraImage pays ~300 ms of tuple marshalling, which
# swamps the CPU/GPU difference this notebook is measuring.
import subprocess
import sys

def numpy_enabled():
    out = subprocess.run([sys.executable, "-c",
                          "import pybullet as p; print(p.isNumpyEnabled())"],
                         capture_output=True, text=True)
    return out.stdout.strip().endswith("1")

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "yourdfpy", "trimesh", "mediapy"], check=False)

if numpy_enabled():
    print("pybullet already has NumPy support")
else:
    print("Rebuilding pybullet from source -- minutes.")
    proc = subprocess.Popen(
        [sys.executable, "-m", "pip", "install", "--force-reinstall", "--no-deps",
         "--no-binary", "pybullet", "--no-build-isolation", "--no-cache-dir", "pybullet"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        if any(k in line for k in ("numpy is", "Building wheel", "Successfully", "error")):
            print("   ", line.rstrip())
    print("   isNumpyEnabled =", numpy_enabled())
    if IN_COLAB:
        print("\n>>> Colab 需要重启运行时（代码执行程序 → 重新启动会话）再从这里继续。")

In [ ]:
# @title 0c. GPU / EGL 可用性
import importlib.util

import pybullet as p
import torch

assert p.isNumpyEnabled(), "上一格装完了吗？Colab 上可能需要先重启运行时"

p.connect(p.DIRECT)
spec = importlib.util.find_spec("eglRenderer")
EGL_OK = spec is not None and p.loadPlugin(spec.origin, "_eglRendererPlugin") >= 0
p.disconnect()

print("torch.cuda        :", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")
print("EGL plugin loads  :", EGL_OK)
if not EGL_OK:
    print("\n没有 GPU 光栅化器，第 1-3 步的 gpu=True 会静默退回 CPU，结论不成立。"
          "\nColab: 代码执行程序 → 更改运行时类型 → GPU，然后从头再跑。")

## 1. 数据：一个真实 episode

用仓库自己的 loader (`core.io.load_depth_data` / `load_extrinsics`)，
拿到的就是 `compute_tracks` / `compute_metrics` 平时吃的那份 `scene_constants`。

为了让整个 notebook 在几分钟内跑完，下面按 `FRAME_STRIDE` 抽帧。
抽帧只影响耗时的绝对值，不影响任何一个对照——两条路吃的是同一批帧。

In [ ]:
# @title 1a. 取 episode
import glob

import numpy as np

FRAME_STRIDE = 4          # @param {type:"integer"}
EPISODE_ID = ""           # @param {type:"string"}   留空则自动挑一个

DEPTH_ROOT = os.path.join(CACHE_DIR, "depth")
EXT_ROOT = os.path.join(CACHE_DIR, "extrinsics")
GCS = "gs://dm-tapnet/mv-tap/droid"

local = sorted(d for d in glob.glob(os.path.join(DEPTH_ROOT, "*"))
               if os.path.exists(os.path.join(d, "robot.npz")))
if not EPISODE_ID:
    if local:
        EPISODE_ID = os.path.basename(local[0])
    elif IN_COLAB:
        EPISODE_ID = "TRI+52ca9b6a+2023-12-13-14h-03m-35s"
    else:
        raise SystemExit(f"No episodes under {DEPTH_ROOT} -- set EPISODE_ID and "
                         "let the next cell fetch it from GCS.")
print("episode:", EPISODE_ID)

ep_cache = os.path.join(DEPTH_ROOT, EPISODE_ID)
if not os.path.exists(os.path.join(ep_cache, "robot.npz")):
    # Colab: pull just this episode. Same layout the pipeline writes, so the
    # repo's own loader reads it unchanged.
    if IN_COLAB:
        from google.colab import auth
        auth.authenticate_user()
    os.makedirs(ep_cache, exist_ok=True)
    os.system(f"gsutil -m cp -r '{GCS}/depth/{EPISODE_ID}/*' '{ep_cache}/'")
    os.makedirs(os.path.join(EXT_ROOT, EPISODE_ID), exist_ok=True)
    os.system(f"gsutil -m cp -r '{GCS}/extrinsics/{EPISODE_ID}/*' "
              f"'{os.path.join(EXT_ROOT, EPISODE_ID)}/'")
print("cached at:", ep_cache)

In [ ]:
# @title 1b. 载入并抽帧
from core.io import get_accelerator, load_depth_data, load_extrinsics

device = get_accelerator()
scene_constants = load_depth_data(EPISODE_ID, DEPTH_ROOT, load_video="first_frame")
scene_state_ds = load_extrinsics(scene_constants, EXT_ROOT)   # dataset/pipeline 外参

def subsample(sc, state, stride):
    """Keep every `stride`-th frame, everywhere at once.

    Every comparison below has to see the same frames, and the per-frame
    arrays live in several places -- slicing them in one function is the only
    way to be sure none is left at full length.
    """
    import copy
    sc = copy.deepcopy(sc)
    state = copy.deepcopy(state)
    for key in ("joint_positions", "gripper_positions", "T_ee_base_all"):
        if key in sc["robot"]:
            sc["robot"][key] = sc["robot"][key][::stride]
    for cam, cd in sc["camera"].items():
        for key in ("raw_depth", "original_raw_depth", "video_rgb", "video_rgb_right"):
            v = cd.get(key)
            if isinstance(v, np.ndarray) and v.ndim >= 3 and len(v) > 1:
                cd[key] = v[::stride]
        state[cam]["extrinsics"] = state[cam]["extrinsics"][::stride]
    return sc, state

N_FRAMES_FULL = len(scene_constants['robot']['joint_positions'])
scene_constants, scene_state_ds = subsample(scene_constants, scene_state_ds, FRAME_STRIDE)

WRIST = scene_constants["meta"]["wrist_serial"]
EXT_CAMS = [c for c in scene_constants["camera"] if c != WRIST]
N_FRAMES = len(scene_constants["robot"]["joint_positions"])
H_IMG, W_IMG = scene_constants["camera"][EXT_CAMS[0]]["raw_depth"][0].shape

print(f"\n帧数     : {N_FRAMES}  (stride {FRAME_STRIDE})")
print(f"分辨率   : {W_IMG}x{H_IMG}")
print(f"外部相机 : {EXT_CAMS}")
print(f"腕部相机 : {WRIST}")

## 2. 第一步：`PyBulletRenderer(gpu=True)` 在真实数据上等价吗

合成 pose 上量到 mask IoU 0.983。真实数据的相机位姿、机器人构型、遮挡关系都不一样，
所以这里在**三台相机 × 多帧真实位姿**上重量一遍。

判据有两条，缺一不可：
- **等价**：mask IoU 要接近 1，深度在真实表面上要亚毫米一致；
- **更快**：每帧耗时要真的降下来。

In [ ]:
# @title 2a. 两条路各渲一遍
import time

from core.physics import PyBulletRenderer

N_AB = min(N_FRAMES, 12)                       # frames per camera for the A/B
FRAMES_AB = np.linspace(0, N_FRAMES - 1, N_AB).astype(int)

def render_all(gpu):
    """Render mask+depth for every (camera, frame) pair under one rasteriser.

    Both renderers cannot coexist -- PyBulletRenderer.__init__ disconnects
    whatever came before -- so each pass runs to completion and keeps its
    results.
    """
    r = PyBulletRenderer(gpu=gpu)
    out, per_frame_ms = {}, []
    for cam in scene_constants["camera"]:
        K = scene_constants["camera"][cam]["K_mat"]
        for t in FRAMES_AB:
            scene_constants["robot"]["joint_positions"][t]
            r.update_robot_pose(scene_constants["robot"]["joint_positions"][t],
                                scene_constants["robot"]["gripper_positions"][t])
            ext = scene_state_ds[cam]["extrinsics"][t]
            t0 = time.perf_counter()
            d = r.render_depth(ext, K, W_IMG, H_IMG)
            per_frame_ms.append((time.perf_counter() - t0) * 1000)
            out[(cam, t)] = (d, r.render_mask(ext, K, W_IMG, H_IMG))
    return r.gpu, out, float(np.median(per_frame_ms))

gpu_ok, RES_GPU, MS_GPU = render_all(True)
_, RES_CPU, MS_CPU = render_all(False)
assert gpu_ok, "EGL 没加载上，下面的对照没有意义"
print(f"CPU {MS_CPU:6.1f} ms/frame   GPU {MS_GPU:6.1f} ms/frame   {MS_CPU / MS_GPU:.1f}x")

In [ ]:
# @title 2b. 逐相机对照表
from scipy import ndimage

print(f"{'camera':<12}{'帧':>4}{'mask IoU':>11}{'mask 覆盖 cpu/gpu':>20}"
      f"{'表面深度中位差':>16}{'表面最大差':>13}")
rows = []
for cam in scene_constants["camera"]:
    ious, med_smooth, max_smooth, cov_c, cov_g = [], [], [], [], []
    for t in FRAMES_AB:
        dc, mc = RES_CPU[(cam, t)]
        dg, mg = RES_GPU[(cam, t)]
        union = (mc | mg).sum()
        if union == 0:
            continue
        ious.append((mc & mg).sum() / union)
        cov_c.append(mc.mean()); cov_g.append(mg.mean())
        both = (dc > 0) & (dg > 0)
        if both.sum() < 100:
            continue
        # "surface" = away from depth steps, where the two rasterisers can
        # legitimately assign a pixel to different surfaces
        hi = ndimage.maximum_filter(np.where(dc > 0, dc, -1e9), size=3)
        lo = ndimage.minimum_filter(np.where(dc > 0, dc, +1e9), size=3)
        smooth = both & ((hi - lo) < 1e-3)
        if smooth.sum() > 100:
            diff = np.abs(dc - dg)[smooth]
            med_smooth.append(np.median(diff)); max_smooth.append(diff.max())
    tag = "wrist" if cam == WRIST else "ext"
    rows.append((cam, tag, np.mean(ious), np.mean(cov_c), np.mean(cov_g),
                 np.median(med_smooth), np.max(max_smooth)))
    print(f"{cam:<12}{len(ious):>4}{np.mean(ious):>11.4f}"
          f"{100 * np.mean(cov_c):>10.2f}% /{100 * np.mean(cov_g):>6.2f}%"
          f"{np.median(med_smooth):>14.2e} m{np.max(max_smooth):>11.2e} m")
print(f"\n每帧耗时: CPU {MS_CPU:.1f} ms  ->  GPU {MS_GPU:.1f} ms  ({MS_CPU / MS_GPU:.1f}x)")

In [ ]:
# @title 2c. 看图：真实画面上的 mask
import matplotlib
import matplotlib.pyplot as plt
from scipy import ndimage

# Categorical slots 1-3 of the reference palette (all-pairs validated:
# worst CVD dE 9.2, worst normal-vision dE 24.0). Figure text stays ASCII.
C_CPU, C_GPU, C_THIRD = "#2a78d6", "#eb6834", "#1baf7a"
SURFACE, INK, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e5e4e0"

cam_show = EXT_CAMS[0]
t_show = 0          # load_video="first_frame": only frame 0 has RGB
rgb = scene_constants["camera"][cam_show].get("first_frame_rgb")
_, mc = RES_CPU[(cam_show, t_show)]
_, mg = RES_GPU[(cam_show, t_show)]

ys, xs = np.where(mc | mg)
box = (slice(max(ys.min() - 30, 0), ys.max() + 30),
       slice(max(xs.min() - 30, 0), xs.max() + 30))

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), facecolor=SURFACE)
base = (rgb[box] / 255.0 if rgb is not None
        else np.ones((box[0].stop - box[0].start, box[1].stop - box[1].start, 3)) * 0.9)
for ax, (m, label, colour) in zip(axes, [(mc[box], "CPU rasteriser", C_CPU),
                                         (mg[box], "EGL rasteriser (gpu=True)", C_GPU),
                                         (None, "disagreement", C_THIRD)]):
    img = base.copy()
    if m is None:
        d = mc[box] ^ mg[box]
        n_px = int(d.sum())
        # a few hundred single pixels on a silhouette are invisible at this
        # scale, so fatten them -- the count in the title is the real one
        img[ndimage.binary_dilation(d, iterations=2)] = matplotlib.colors.to_rgb(C_THIRD)
        label = f"disagreement: {n_px} px (dilated 2px to be visible)"
    else:
        img[m] = 0.35 * img[m] + 0.65 * np.array(matplotlib.colors.to_rgb(colour))
    ax.imshow(np.clip(img, 0, 1))
    ax.set_title(label, fontsize=9.5, color=INK, pad=8)
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_color(GRID)
fig.suptitle(f"{cam_show}, frame {t_show} -- same pose, two rasterisers",
             fontsize=11, color=INK, y=1.02)
fig.tight_layout()
plt.show()

## 3. 第二步：点云生成能降到多低的分辨率

外参优化只需要 `MAX_ROBOT_PTS`（仓库里是 2000）个机器人表面点，
但 `get_foreground_robot_points` 是在**全分辨率**渲染完再丢掉 95%。
降分辨率只要把 `K` 同比缩放，反投影出的世界点坐标不变。

两个问题：**点还够不够**，以及**点云是不是同一朵**。

In [ ]:
# @title 3a. 分辨率 vs 点数 vs 耗时
from compute_metrics import get_foreground_robot_points

MAX_ROBOT_PTS = 2000
SCALES = [1.0, 0.75, 0.5, 0.25, 0.125]
cam = EXT_CAMS[0]
K_full = scene_constants["camera"][cam]["K_mat"]

def scaled(K, s):
    """Intrinsics for a render s times smaller. Unprojection is unchanged."""
    K2 = K.copy()
    K2[:2] *= s
    return K2

r_gpu = PyBulletRenderer(gpu=True)
SWEEP = {}
print(f"{'scale':>7}{'render':>12}{'robot px':>11}{'>=2000?':>9}{'ms/frame':>11}")
for s in SCALES:
    w, h = int(W_IMG * s), int(H_IMG * s)
    K_s = scaled(K_full, s)
    counts, times = [], []
    for t in FRAMES_AB[:6]:
        r_gpu.update_robot_pose(scene_constants["robot"]["joint_positions"][t],
                                scene_constants["robot"]["gripper_positions"][t])
        ext = scene_state_ds[cam]["extrinsics"][t]
        r_gpu.render_depth(ext, K_s, w, h)              # warm
        t0 = time.perf_counter()
        d = r_gpu.render_depth(ext, K_s, w, h)
        times.append((time.perf_counter() - t0) * 1000)
        counts.append(int((d > 0).sum()))
    SWEEP[s] = (np.median(counts), np.median(times), w, h)
    ok = "yes" if np.median(counts) >= MAX_ROBOT_PTS else "NO"
    print(f"{s:>7.3f}{f'{w}x{h}':>12}{int(np.median(counts)):>11}{ok:>9}"
          f"{np.median(times):>11.2f}")

In [ ]:
# @title 3b. 降分辨率后还是同一朵点云吗
# One-sided nearest-neighbour distance from the downscaled cloud to the
# full-resolution one: if the low-res points sit on the same surface, every
# one of them has a full-res neighbour a fraction of a millimetre away.
import torch

cam = EXT_CAMS[0]
t = FRAMES_AB[len(FRAMES_AB) // 2]
r_gpu.update_robot_pose(scene_constants["robot"]["joint_positions"][t],
                        scene_constants["robot"]["gripper_positions"][t])
ext = scene_state_ds[cam]["extrinsics"][t]
obs = scene_constants["camera"][cam]["raw_depth"][t].astype(np.float32)

ref = get_foreground_robot_points(ext, K_full, obs, r_gpu, device, max_pts=8000)
print(f"{'scale':>7}{'点数':>8}{'到全分辨率点云的最近邻距离':>30}")
for s in SCALES:
    w, h = int(W_IMG * s), int(H_IMG * s)
    if SWEEP[s][0] < MAX_ROBOT_PTS:
        print(f"{s:>7.3f}{'--':>8}   点数不足，跳过")
        continue
    pts = get_foreground_robot_points(ext, scaled(K_full, s),
                                      np.zeros((h, w), np.float32), r_gpu,
                                      device, max_pts=MAX_ROBOT_PTS)
    nn = torch.cdist(pts[None], ref[None])[0].min(dim=1)[0]
    print(f"{s:>7.3f}{len(pts):>8}   中位 {nn.median().item() * 1000:6.2f} mm"
          f" | p99 {torch.quantile(nn, 0.99).item() * 1000:6.2f} mm")

## 4. 第三步：外参优化，yourdfpy vs PyBullet

现在两条路都在同一个 episode 上跑一遍，从**同一个初始外参**（数据集自带的）出发：

| | 点云来源 | 可见性 | 损失 |
|---|---|---|---|
| **A（现状）** | `TensorRobotRenderer`：yourdfpy FK + mesh 表面采样 | 靠法线做 front-face culling 近似 | `compute_extrinsics.compute_robot_loss`（带法线、带 tolerance） |
| **B（复活）** | `PyBulletRenderer`：光栅化后反投影 | 光栅化器天然只给可见面 | `compute_metrics.compute_robot_loss_batched`（无法线、无 culling） |

两条路的损失函数不同，所以**收敛 loss 之间不可比**。裁判用第三方：
`compute_metrics.evaluate_extrinsics`——它对任何 `scene_state` 都用同一套
PyBullet 渲染 + Chamfer 来打分，三个状态（初始 / A / B）用同一把尺子。

> 注意 B 的点云来源和裁判共用一个渲染器。这不是循环论证（裁判量的是外参和**观测深度**
> 的一致性，不是和渲染的一致性），但也不是完全中立，读结论时要记得这一点。

In [ ]:
# @title 4a. 方法 A：现状的 yourdfpy 路径
import copy
import time

from compute_extrinsics import phase2_per_camera_alignment
from core.physics import TensorRobotRenderer

t0 = time.perf_counter()
tensor_renderer = TensorRobotRenderer(device=device)
state_A = phase2_per_camera_alignment(scene_constants, tensor_renderer, scene_state_ds)
T_A = time.perf_counter() - t0
print(f"\n方法 A 用时 {T_A:.1f} s")

In [ ]:
# @title 4b. 方法 B：PyBullet 点云 + 你当年的损失
import torch.optim as optim

from compute_metrics import (compute_robot_loss_batched, compute_wrist_loss_batched,
                             get_foreground_gripper_points, get_foreground_robot_points)
from core.geometry import make_T

OUTER, INNER = 3, 167          # 3 x 167 ~= the 500 Adam steps method A takes
RENDER_SCALE = 0.5             # @param  step 2 says half resolution is plenty

def optimise_camera_pybullet(cam, pb, T_init_np):
    """Outer loop re-renders the point cloud under the current estimate;
    inner loop is pure torch on those (now constant) points."""
    is_wrist = (cam == WRIST)
    K_np = scene_constants["camera"][cam]["K_mat"]
    K_t = torch.tensor(K_np, dtype=torch.float32, device=device)
    T_init_t = torch.tensor(T_init_np, dtype=torch.float32, device=device)
    w_r, h_r = int(W_IMG * RENDER_SCALE), int(H_IMG * RENDER_SCALE)
    K_render = K_np.copy(); K_render[:2] *= RENDER_SCALE
    dummy = np.zeros((h_r, w_r), np.float32)      # only its shape is read

    d_ext = torch.zeros(6, requires_grad=True, device=device)
    optimizer = optim.Adam([d_ext], lr=0.001)
    loss_val = float("nan")

    for outer in range(OUTER):
        with torch.no_grad():
            T_cur = (T_init_t @ make_T(d_ext, device)).cpu().numpy()
        cache_X, cache_obs = [], []
        for t in range(N_FRAMES):
            pb.update_robot_pose(scene_constants["robot"]["joint_positions"][t],
                                 scene_constants["robot"]["gripper_positions"][t])
            obs = scene_constants["camera"][cam]["raw_depth"][t].astype(np.float32)
            if is_wrist:
                T_cw = scene_constants["robot"]["T_ee_base_all"][t] @ T_cur
                pts = get_foreground_gripper_points(T_cw, K_render, dummy, pb, device)
                if pts is None:
                    continue
                cache_X.append(torch.tensor(
                    (T_cur @ pts)[:3, :].T, dtype=torch.float32,
                    device=device))
            else:
                pts = get_foreground_robot_points(T_cur, K_render, dummy, pb, device)
                if pts is None:
                    continue
                cache_X.append(pts)
            cache_obs.append(torch.tensor(obs, dtype=torch.float32, device=device)[None])
        if not cache_X:
            print(f"    [WARN] {cam}: no points at outer {outer}")
            return T_init_np, float("nan")
        batch_X, batch_obs = torch.stack(cache_X), torch.stack(cache_obs)

        for _ in range(INNER):
            optimizer.zero_grad()
            T_opt = T_init_t @ make_T(d_ext, device)
            loss = (compute_wrist_loss_batched(batch_X, T_opt, K_t, batch_obs)
                    if is_wrist else
                    compute_robot_loss_batched(batch_X, T_opt, K_t, batch_obs))
            loss.backward()
            optimizer.step()
            loss_val = loss.item()
        print(f"    outer {outer + 1}/{OUTER} | frames {len(cache_X)} | loss {loss_val:.4f}")

    with torch.no_grad():
        return (T_init_t @ make_T(d_ext, device)).cpu().numpy(), loss_val

t0 = time.perf_counter()
state_B = copy.deepcopy(scene_state_ds)
for cam in scene_constants["camera"]:
    print(f"  [{cam}] {'wrist' if cam == WRIST else 'external'}")
    T_fin, _ = optimise_camera_pybullet(cam, r_gpu, scene_state_ds[cam]["base_extrinsic"])
    state_B[cam]["base_extrinsic"] = T_fin
    state_B[cam]["extrinsics"] = (scene_constants["robot"]["T_ee_base_all"] @ T_fin
                                  if cam == WRIST else np.tile(T_fin, (N_FRAMES, 1, 1)))
T_B = time.perf_counter() - t0
print(f"\n方法 B 用时 {T_B:.1f} s  (渲染分辨率 {RENDER_SCALE}x)")

In [ ]:
# @title 4c. 中立裁判
from compute_metrics import evaluate_extrinsics

SCORES = {}
for name, st in (("初始（数据集）", scene_state_ds), ("A: yourdfpy", state_A),
                 ("B: pybullet", state_B)):
    SCORES[name] = evaluate_extrinsics(scene_constants, st, device, pb_renderer=r_gpu)

keys = [("robot_loss_cam1", "robot cam1"), ("robot_loss_cam2", "robot cam2"),
        ("robot_loss_wrist", "robot wrist"), ("chamfer_total", "chamfer"),
        ("bg_overlap_pct", "bg overlap %")]
print(f"{'':<16}" + "".join(f"{lab:>14}" for _, lab in keys))
for name, m in SCORES.items():
    print(f"{name:<16}" + "".join(f"{m.get(k, float('nan')):>14.4f}" for k, _ in keys))
print(f"\n耗时  A {T_A:.1f} s   B {T_B:.1f} s   ({T_A / T_B:.2f}x)")

print("\n两条路解出来的外参差多少:")
for cam in scene_constants["camera"]:
    dT = np.linalg.inv(state_A[cam]["base_extrinsic"]) @ state_B[cam]["base_extrinsic"]
    shift = np.linalg.norm(dT[:3, 3]) * 1000
    rot = np.degrees(np.arccos(np.clip((np.trace(dT[:3, :3]) - 1) / 2, -1, 1)))
    print(f"  {cam:<12} 平移 {shift:7.2f} mm   旋转 {rot:6.3f}°")

In [ ]:
# @title 4d. 看图：三个外参投到真实画面上
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), facecolor=SURFACE)
cam = EXT_CAMS[0]
K = scene_constants["camera"][cam]["K_mat"]
rgb = scene_constants["camera"][cam]["first_frame_rgb"]
r_gpu.update_robot_pose(scene_constants["robot"]["joint_positions"][0],
                        scene_constants["robot"]["gripper_positions"][0])

panels = [("dataset init", scene_state_ds, C_THIRD),
          ("A: yourdfpy", state_A, C_CPU),
          ("B: pybullet", state_B, C_GPU)]
masks = {lab: r_gpu.render_mask(st[cam]["extrinsics"][0], K, W_IMG, H_IMG)
         for lab, st, _ in panels}
ys, xs = np.where(np.logical_or.reduce(list(masks.values())))
box = (slice(max(ys.min() - 30, 0), ys.max() + 30),
       slice(max(xs.min() - 30, 0), xs.max() + 30))

for ax, (label, _, colour) in zip(axes, panels):
    m = masks[label][box]
    img = rgb[box] / 255.0
    edge = m ^ ndimage.binary_erosion(m, iterations=2)
    img = img.copy()
    img[edge] = matplotlib.colors.to_rgb(colour)
    ax.imshow(np.clip(img, 0, 1))
    ax.set_title(label, fontsize=9.5, color=INK, pad=8)
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_color(GRID)
fig.suptitle(f"{cam}, frame 0 -- rendered robot outline over the real image",
             fontsize=11, color=INK, y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# @title 4e. 汇总图
fig, axes = plt.subplots(1, 3, figsize=(12, 2.9), facecolor=SURFACE)
labels = ["dataset init", "A: yourdfpy", "B: pybullet"]
colours = [C_THIRD, C_CPU, C_GPU]
panels = [
    ("robot depth loss, cam1  (lower is better)",
     [SCORES[n].get("robot_loss_cam1", np.nan) for n in SCORES], "%.4f"),
    ("chamfer total  (lower is better)",
     [SCORES[n].get("chamfer_total", np.nan) for n in SCORES], "%.4f"),
    ("wall clock, seconds  (init is free)", [0.0, T_A, T_B], "%.0f s"),
]
for ax, (title, vals, fmt) in zip(axes, panels):
    y = np.arange(3)[::-1]
    ax.barh(y, vals, height=0.34, color=colours, zorder=3)
    span = max(v for v in vals if np.isfinite(v)) or 1.0
    for yi, v in zip(y, vals):
        if np.isfinite(v):
            ax.text(v + span * 0.03, yi, fmt % v, va="center", fontsize=9.5, color=INK)
    ax.set_yticks(y); ax.set_yticklabels(labels, fontsize=9, color=MUTED)
    ax.set_title(title, fontsize=9.5, color=INK, loc="left", pad=10)
    ax.set_xlim(0, span * 1.3)
    ax.set_facecolor(SURFACE); ax.xaxis.set_visible(False)
    for side in ("top", "right", "bottom"):
        ax.spines[side].set_visible(False)
    ax.spines["left"].set_color(GRID); ax.tick_params(length=0)
fig.tight_layout()
plt.show()

## 5. 当年为什么慢，现在还慢不慢

方法 B 上面用的是 `gpu=True` + 半分辨率。把这两条都退回去，就是 v50 那份代码的处境——
它的 `PyBulletRenderer_Robotiq` 里写的是 `find_spec('eglRendererPlugin')`，
一个不存在的模块名，所以插件从来没加载过，全程 CPU 全分辨率。

下面按 v50 的循环规模（Stage 2 每台相机 2 个基底 × OUTER 5 轮，加上 Stage 3/4 各一遍预计算）
把渲染预算算出来。

In [ ]:
MS_CPU_FULL = MS_CPU                     # measured in 2a, full resolution
MS_GPU_HALF = SWEEP[0.5][1]              # measured in 3a
FULL_FRAMES = N_FRAMES_FULL              # the episode without subsampling

# v50's loop: stage 2 external = 2 cams x 2 base candidates x OUTER 5,
# stage 2 wrist = 5, stage 3 and stage 4 precompute 3 clouds per frame each.
renders = FULL_FRAMES * (2 * 2 * 5 + 5 + 3 + 3)
print(f"一个 episode（{FULL_FRAMES} 帧，不抽帧）的渲染次数: {renders}")
print(f"{'配置':<34}{'每帧':>10}{'总计':>12}")
for label, ms in (("v50 当年: CPU + 全分辨率", MS_CPU_FULL),
                  ("只修插件名: GPU + 全分辨率", MS_GPU),
                  ("再加降分辨率: GPU + 0.5x", MS_GPU_HALF)):
    total = renders * ms / 1000
    print(f"{label:<34}{ms:>8.1f}ms{total / 60:>10.1f} min")
print(f"\n端到端提速: {MS_CPU_FULL / MS_GPU_HALF:.0f}x")

## 结论

三步逐条对照，都在真实 episode 上量过：

**第 1 步 —— 过。** 三台相机、12 帧真实位姿，`gpu=True` 与 CPU 的 mask IoU **0.994–0.995**
（比合成 pose 上的 0.983 还高，因为机器人在画面里占 16–20%，边界像素占比更小），
表面深度中位差 **0.8–1.1e-4 m**，亚毫米。速度 **220 ms → 28 ms，7.7x**——
比合成场景的 4.4 倍更大，因为 CPU 光栅化器的开销随画面里的三角形数量涨，而 EGL 基本持平。

一个保留意见：某台相机上"光滑表面"像素的**最大**差达到 7.2e-2 m。中位数是 0.08 mm，
所以这是极少数细结构像素上的判定分歧，但它说明 3x3 邻域判据不足以完全隔离边界效应。

**第 2 步 —— 过，而且比预想的宽松。** 这个 episode 一直降到 160x90 都还有 2592 个机器人像素
（够 2000 的门槛）。点云等价性用"到全分辨率点云的最近邻距离"量：
scale 1.0 自己对自己是 **1.92 mm**（两次独立采样 2000 点的噪声地板），
0.5x 是 **2.08 mm**，0.125x 才涨到 2.80 mm。也就是说降到一半，误差还埋在采样噪声里。
耗时 28.05 → 7.74 ms。

**第 3 步 —— 可行，但不是碾压。** 同一初始外参出发，中立裁判打分：

| | robot cam1 | robot cam2 | robot wrist | chamfer |
|---|---|---|---|---|
| 初始（数据集） | 0.0212 | 0.0156 | 0.0177 | 0.0863 |
| A: yourdfpy | **0.0177** | **0.0151** | 0.0113 | 0.0910 |
| B: pybullet | 0.0183 | 0.0168 | **0.0098** | 0.0917 |

B 在腕部相机上明显更好（0.0098 vs 0.0113），外部相机上略逊；两条路解出的外参相差
1.8–5.4 mm / 0.09–2.6°。耗时 14.1 s vs 12.7 s，基本打平。

所以**"统一到 PyBullet"在这个 episode 上是站得住的，但理由不是精度碾压，也不是速度**——
是少一个依赖、可见性不用再靠法线近似。腕部那个优势符合预期：夹爪自遮挡严重，
front-face culling 近似得最吃力的地方，正是光栅化器免费给对的地方。

两条都要注意：**A 和 B 都让 chamfer 略微变差**（0.0863 → 0.091），说明 Phase 2 单独优化机器人
对齐会牺牲一点环境缝合，这是 Phase 3 联合优化存在的理由——本 notebook 没有覆盖 Phase 3。

### 还需要做什么

1. **多 episode**。以上全部基于一个 episode。B 在腕部的优势、A 在外部相机的优势
   是不是稳定的，得跑 `episodes_eval50.txt` 那批才知道。
2. **Phase 3 联合优化**没测。B 要真的替换 A，得连 `phase3_global_joint_alignment` 一起换。
3. **`compute_tracks` 的端到端影响**没测。第 1 步证明了单帧 mask IoU 0.994，
   但 0.6% 的边界像素分歧会不会累积到轨迹上，要跑完整 Stage 3 才知道。
4. 上面那个 7.2e-2 m 的最大差值得单独看一眼是哪个像素。